# Image Classifier
In this notebook we are going to combine the techniques we learned in the previous 2 notebooks:

- How to use the camera
- How to run the neural network on our Raspberry Pi

We are going to take the images from our camera and run them through our neural network.

# Setup
Run the code below, this will import the libraries we need to use the camera and our neural network.

_Run the code below_

In [ ]:
# RUN ME
!big-display off
import pickle
from IPython.display import display
import numpy as np
from PIL import Image
import os
import json
import pickle
os.environ["DISPLAY"] = ":0"

import torch
from torch import nn
from torchvision import models, transforms
from executorch.runtime import Runtime
from IPython.display import display
import numpy as np
from picamera2 import Picamera2, Preview, Platform, MappedArray
import cv2

# Load our Model
Just like in the last notebook we are going to use `executorch`'s `Runtime` class to load our `model.pte` file:

_Run the code below_

In [ ]:
runtime = Runtime.get()
program = runtime.load_program("./model-files/model.pte")
model = program.load_method("forward")

# Load our Classes
Similarly we will load our class names with the `json.load` function:

_Run the code below_

In [ ]:
# RUN ME
classes = []
with open("./model-files/classes.json", "r") as f:
    classes = json.load(f)

print("Loaded classes", classes)

# Setup Transforms
Finally, we will set up those transforms we need to normalize our images:

_Run the code below_

In [ ]:
# RUN ME
mobilenet_mean = [0.485, 0.456, 0.406]
mobilenet_std = [0.229, 0.224, 0.225]

validation_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mobilenet_mean,
        std=mobilenet_std,
    )
])

unnormalize_transform = transforms.Normalize(
    mean=[-m / s for m, s in zip(mobilenet_mean, mobilenet_std)],
    std=[1 / s for s in mobilenet_std]
)

# Setup Camera
Next we will set up our camera.

_Run the code below_

In [ ]:
# RUN ME
cam = Picamera2()
cam_resolution = (224, 224)

# Image annotation code
image_annotations = []

#  Text properties
colour = (0, 255, 0)
text_height = 30
origin = (30, cam_resolution[1] - int(text_height * 1.5))
font = cv2.FONT_HERSHEY_SIMPLEX
scale = 0.75
thickness = 2

# Function which writes whatever is in the image_annotations variable on the screen
def annotate_image(request):
    with MappedArray(request, "main") as m:
        for i, line in enumerate(image_annotations):
            # We do some math to make sure each line of text is below the next
            cv2.putText(m.array, line, (origin[0], origin[1] + (i * text_height)), font, scale, colour, thickness)

# Setup camera
config = cam.create_preview_configuration(main={
    "size": cam_resolution,
    "format": "BGR888",
}, display="main")
cam.configure(config)
cam.set_controls({"FrameRate": 36})
cam.pre_callback = annotate_image
cam.start_preview(Preview.QT)
cam.start()

Now the display on your Raspberry Pi should be showing what the camera is capturing.

If you get any errors then try the troubleshooting steps from the camera preview notebook.

# Run Our Neural Network
Now we are going to run our neural network in a loop:

- Get an image from the camera
- Run the image through our neural network
- Display what our neural network thinks is in the image on the screen

The code in the cell below will keep running in a loop until you click the stop button. Once you run the code loop at the display and see how your neural network performs.

- Point your camera at the different objects you trained your neural network on
- Does it identify classes correctly?
- Does it fail to identify any classes?
  - If it fails, can you figure out why?
  - Did you have training data that looks like the object it fails to identify?

_Run the code below_

In [ ]:
# RUN ME
# Loop forever
# Or until you stop this cell by hitting the stop button
while True:
    # Get an image of what is being recorded by the camera
    img = cam.capture_image("main")

    # Normalize the image so our neural network can use it as input
    input_tensor = validation_transforms(img)
    input_batch = input_tensor.unsqueeze(0)

    # Run our image through the neural network
    batch_output = model.execute(input_batch)

    # Figure out which class is most likely in the image
    output = batch_output[0][0]

    output_class_idx = torch.tensor(output).argmax().item()
    output_class = classes[output_class_idx]
    
    normalized_output = nn.Softmax()(output)
    output_class_pct = normalized_output[output_class_idx].item()

    # Set the image_annotations variable to say what the model detected
    image_annotations = [
        output_class,
        f"{round(output_class_pct*100, 2):0.2f}%",
    ]

Finally, stop the preview and shut the camera down.

In [ ]:
cam.stop_preview()
cam.stop()